# Telecom Customer Churn Analysis
## 04 — SQL Analysis

### Objective
Use SQL to answer business questions about customer churn and retention.
Understand which customer groups have the highest churn rates and turn the results into stronger business insights.

In [1]:
import pandas as pd
import sqlite3

In [2]:
# Load the cleaned dataset

dataset = r"C:\Users\HomePC\Desktop\Data Analysis Portfolio\telecom_customer_churn_analysis\data\telecom_customer_churn_clean.csv"
df_clean = pd.read_csv(dataset)

In [3]:
# Create a SQLite database connection

conn = sqlite3.connect("../data/telecom_churn.db")

In [4]:
# Load the data into a SQL table

df_clean.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded successfully!")

Data loaded successfully!


In [17]:
# Confirm  total count of customers

pd.read_sql_query(
    "SELECT COUNT(*) AS total_customers FROM customers;",
    conn
)

,total_customers
0,7043


In [6]:
# Query 1: How many customers are in each status?

query = """
SELECT
    "Customer Status",
    COUNT(*) AS customer_count
FROM customers
GROUP BY "Customer Status"
ORDER BY customer_count DESC;
"""

df_result = pd.read_sql_query(query, conn)

df_result

,Customer Status,customer_count
0,Stayed,4720
1,Churned,1869
2,Joined,454


In [18]:
# Alternative method

pd.read_sql_query(
    """SELECT
        "Customer Status",
        COUNT(*) AS customer_count
    FROM customers
    GROUP BY "Customer Status"
    ORDER BY customer_count DESC;""",
    conn
)

,Customer Status,customer_count
0,Stayed,4720
1,Churned,1869
2,Joined,454


In [20]:
# Query 2: What percentage of each contract group churned?

pd.read_sql_query(
    """
    SELECT
        Contract,
        COUNT(*) AS total_customers,
        SUM(
            CASE
                WHEN "Customer Status" = 'Churned' THEN 1
                ELSE 0
            END
        ) AS churned_customers,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS churn_percentage
    FROM customers
    GROUP BY Contract
    ORDER BY churn_percentage DESC;    
    """,
    conn
)

,Contract,total_customers,churned_customers,churn_percentage
0,Month-to-Month,3610,1655,45.84
1,One Year,1550,166,10.71
2,Two Year,1883,48,2.55


In [21]:
# Query 3: Which internet types have the highest churn percentage?

pd.read_sql_query(
    """
SELECT
    "Internet Type",
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Customer Status" = 'Churned' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_percentage
    FROM customers
    WHERE "Internet Service" = 'Yes'
    GROUP BY "Internet Type"
    ORDER BY churn_percentage DESC;
    """,
    conn
)

,Internet Type,total_customers,churned_customers,churn_percentage
0,Fiber Optic,3035,1236,40.72
1,Cable,830,213,25.66
2,DSL,1652,307,18.58


In [22]:
# Query 4: What percentage of each tenure group churned?

pd.read_sql_query(
    """
SELECT
    CASE
        WHEN "Tenure in Months" <= 12 THEN '0-12 months'
        WHEN "Tenure in Months" <= 24 THEN '13-24 months'
        WHEN "Tenure in Months" <= 48 THEN '25-48 months'
        ELSE '49-72 months'
    END AS tenure_group,
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) AS REAL
        ) * 100 / COUNT(*),
        2
    ) AS churn_percentage
    FROM customers
    GROUP BY tenure_group
    ORDER BY
        CASE tenure_group
            WHEN '0-12 months' THEN 1
            WHEN '13-24 months' THEN 2
            WHEN '25-48 months' THEN 3
            ELSE 4
        END;
    """,
    conn
)

,tenure_group,total_customers,churned_customers,churn_percentage
0,0-12 months,2186,1037,47.44
1,13-24 months,1024,294,28.71
2,25-48 months,1594,325,20.39
3,49-72 months,2239,213,9.51


In [23]:
# Query 5: What percentage of each payment method churned?

pd.read_sql_query(
    """
SELECT
    "Payment Method",
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN "Customer Status" = 'Churned' THEN 1
                    ELSE 0
                END
            ) AS REAL
        ) * 100 / COUNT(*),
        2
    ) AS churn_percentage
FROM customers
GROUP BY "Payment Method"
ORDER BY churn_percentage DESC;
""",
    conn
)

,Payment Method,total_customers,churned_customers,churn_percentage
0,Mailed Check,385,142,36.88
1,Bank Withdrawal,3909,1329,34.00
2,Credit Card,2749,398,14.48


### Summary of findings

1. **Internet Type:** Fiber Optic customers had the highest observed churn percentage (40.72%), followed by Cable customers (25.66%) and DSL customers (18.58%). This analysis included only customers with internet service.

2. **Contract Type:** Month-to-month customers had an observed churn percentage of 45.84%, compared with 10.71% for one-year customers and 2.55% for two-year customers.

3. **Customer Tenure:** Customers with 0–12 months of tenure had the highest observed churn percentage (47.44%). The percentage declined across the longer-tenure groups, reaching 9.51% among customers with 49–72 months of tenure.

4. **Payment Method:** The SQL analysis identified differences in observed churn percentages across payment methods. Customers paying by mailed check had the highest churn percentage at 36.88%, followed by customers using bank withdrawal at 34.00%. Customers paying by credit card had the lowest churn percentage at 14.48%.

    Although mailed-check customers had the highest churn percentage, bank withdrawal customers accounted for the largest number of churned customers (1,329), reflecting the larger size of that group.


In [25]:
# Query 6: How are Contract Type and Tenure related to Churn


query = """
SELECT
    Contract,
    CASE
        WHEN "Tenure in Months" <= 12 THEN '0-12 months'
        WHEN "Tenure in Months" <= 24 THEN '13-24 months'
        WHEN "Tenure in Months" <= 48 THEN '25-48 months'
        ELSE '49-72 months'
    END AS tenure_group,
    COUNT(*) AS total_customers,
    SUM(CASE
        WHEN "Customer Status" = 'Churned' THEN 1
        ELSE 0
    END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE
            WHEN "Customer Status" = 'Churned' THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS churn_rate_pct
FROM customers
GROUP BY Contract, tenure_group
ORDER BY Contract, tenure_group;
"""

contract_tenure = pd.read_sql_query(query, conn)
contract_tenure


,Contract,tenure_group,total_customers,churned_customers,churn_rate_pct
0,Month-to-Month,0-12 months,1913,1024,53.53
1,Month-to-Month,13-24 months,686,278,40.52
2,Month-to-Month,25-48 months,715,264,36.92
3,Month-to-Month,49-72 months,296,89,30.07
4,One Year,0-12 months,146,13,8.90
5,One Year,13-24 months,215,16,7.44
6,One Year,25-48 months,546,55,10.07
7,One Year,49-72 months,643,82,12.75
8,Two Year,0-12 months,127,0,0.00
9,Two Year,13-24 months,123,0,0.00


### Notes:

The combined analysis of contract type and tenure shows that month-to-month customers have substantially higher churn rates than customers on one-year or two-year contracts. The highest observed churn rate was 53.53% among month-to-month customers with 0–12 months of tenure. Churn remained at 30.07% even among month-to-month customers with 49–72 months of tenure.

By comparison, two-year contract customers had low observed churn across all tenure groups, ranging from 0% to 3.23%.



In [26]:
# Query 7: How are Contract Type and Avg. Monthly Charge related to Churn


query = """
SELECT
    Contract,
    "Customer Status",
    COUNT(*) AS customer_count,
    ROUND(AVG("Monthly Charge"), 2) AS avg_monthly_charge
FROM customers
WHERE "Monthly Charge" IS NOT NULL
GROUP BY Contract, "Customer Status"
ORDER BY Contract, "Customer Status";
"""

charges_by_contract = pd.read_sql_query(query, conn)
charges_by_contract


,Contract,Customer Status,customer_count,avg_monthly_charge
0,Month-to-Month,Churned,1625,73.20
1,Month-to-Month,Joined,403,42.98
2,Month-to-Month,Stayed,1521,63.44
3,One Year,Churned,166,85.05
4,One Year,Joined,23,37.30
5,One Year,Stayed,1331,63.86
6,Two Year,Churned,48,86.78
7,Two Year,Joined,22,57.70
8,Two Year,Stayed,1784,61.88


### Notes:

Across month-to-month, one-year, and two-year contracts, customers who churned had higher average monthly charges than customers who stayed. The difference ranged from 9.76 units for month-to-month contracts to 24.90 units for two-year contracts.

In [27]:
# Query 8: Churn Category and Churn Reasons Analysis


query = """
SELECT
    "Churn Category",
    "Churn Reason",
    COUNT(*) AS customer_count,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*)
         FROM customers
         WHERE "Customer Status" = 'Churned'),
        2
    ) AS pct_of_all_churned
FROM customers
WHERE "Customer Status" = 'Churned'
GROUP BY "Churn Category", "Churn Reason"
ORDER BY customer_count DESC;
"""

churn_reasons = pd.read_sql_query(query, conn)
churn_reasons.head(15)


,Churn Category,Churn Reason,customer_count,pct_of_all_churned
0,Competitor,Competitor had better devices,313,16.75
1,Competitor,Competitor made better offer,311,16.64
2,Attitude,Attitude of support person,220,11.77
3,Other,Don't know,130,6.96
4,Competitor,Competitor offered more data,117,6.26
5,Competitor,Competitor offered higher download speeds,100,5.35
6,Attitude,Attitude of service provider,94,5.03
7,Price,Price too high,78,4.17
8,Dissatisfaction,Product dissatisfaction,77,4.12
9,Dissatisfaction,Network reliability,72,3.85


### Notes: 
### Competitor Offers and Customer Experience Are Major Reported Churn Reasons

Competitor-related reasons dominate the leading churn reasons in the dataset. Four reported reasons—better devices, better offers, more data, and higher download speeds—account for 841 customers, approximately 45.0% of all churned customers.

Support experience is another important area. The two attitude-related reasons shown in the analysis account for 314 customers, approximately 16.8% of all churned customers.

Pricing and service reliability also appear among the leading reported concerns.

**Recommendations**

1. **Review competitive positioning:** Compare the company's device options, data allowances, internet speeds, and offers with those of competitors.
2. **Improve support quality:** Review customer-service feedback, coach support staff, and monitor customer satisfaction after support interactions.
3. **Investigate pricing concerns:** Review billing clarity, long-distance charges, and extra-data charges to identify potential sources of dissatisfaction.
4. **Monitor service reliability:** Investigate network reliability complaints and prioritize improvements supported by service-performance data.


In [28]:
# Query 9: Group Churn Category to find percentage of all churned customers

query = """
SELECT
    "Churn Category",
    COUNT(*) AS customer_count,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*)
         FROM customers
         WHERE "Customer Status" = 'Churned'),
        2
    ) AS pct_of_all_churned
FROM customers
WHERE "Customer Status" = 'Churned'
GROUP BY "Churn Category"
ORDER BY customer_count DESC;
"""

churn_categories = pd.read_sql_query(query, conn)
churn_categories


,Churn Category,customer_count,pct_of_all_churned
0,Competitor,841,45.00
1,Dissatisfaction,321,17.17
2,Attitude,314,16.80
3,Price,211,11.29
4,Other,182,9.74


## Summary

### Key findings

1. **Contract type and tenure:** Month-to-month customers had the highest churn rates, especially those with 0–12 months of tenure, at 53.53%. Two-year contract customers had comparatively low churn across all tenure groups.
2. **Monthly charges:** Customers who churned had higher average monthly charges than customers who stayed across all three contract types. This is an association, not proof that higher prices cause churn.
3. **Reported reasons:** Competitor-related reasons accounted for 841 customers (45.00% of churned customers). Dissatisfaction accounted for 321 (17.17%), attitude-related reasons for 314 (16.80%), price for 211 (11.29%), and other reasons for 182 (9.74%).
4. **Competitive pressure:** Better devices and better offers were the two most frequently reported individual churn reasons in the results reviewed.
5. **Customer experience:** Support attitude, product and service dissatisfaction, and network reliability were additional areas worth investigating.

### Recommended business actions

- Investigate the needs and experiences of newer month-to-month customers.
- Compare the company's devices, data allowances, internet speeds, and offers with competitors.
- Review customer-support feedback and service-reliability complaints.
- Investigate billing and pricing concerns, including long-distance and extra-data charges.
- Measure whether any retention initiatives actually improve customer outcomes.



# Skills practiced

- SQL aggregation, 
- Conditional grouping,
- Subqueries, 
- Percentage calculations, 
- Segmentation, 
- Interpretation of churn patterns, and 
- Translating analytical findings into business recommendations.